## **Feature Engineering**

In [1]:
# Bibliotecas
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.preprocessing import StandardScaler

# Carregar dados
clientes = pd.read_csv('../data/raw/clientes.csv')
transacoes = pd.read_csv('../data/raw/transacoes.csv')
transacoes['data_compra'] = pd.to_datetime(transacoes['data_compra'])

print("Dados carregados")
print(f"Clientes: {clientes.shape}")
print(f"Transações: {transacoes.shape}")

Dados carregados
Clientes: (10000, 6)
Transações: (72294, 5)


## **Features de Recência, Frequência e Valor (RFV):**

In [2]:
data_ref = transacoes['data_compra'].max() + pd.Timedelta(days=1)

rfv = transacoes.groupby('id_cliente').agg({
    'data_compra': lambda x: (data_ref - x.max()).days,
    'id_cliente': 'count',
    'valor': ['sum', 'mean', 'std']
}).round(2)

rfv.columns = ['recencia_dias', 'frequencia_compras', 'valor_total', 'ticket_medio', 'std_valor']
rfv = rfv.reset_index()

# Merge com clientes
df = clientes.merge(rfv, on='id_cliente', how='left')

# Tratar nulos (clientes sem compras)
df['recencia_dias'] = df['recencia_dias'].fillna(999)
df['frequencia_compras'] = df['frequencia_compras'].fillna(0)
df['valor_total'] = df['valor_total'].fillna(0)
df['ticket_medio'] = df['ticket_medio'].fillna(0)
df['std_valor'] = df['std_valor'].fillna(0)
df['cliente_sem_compra'] = (df['frequencia_compras'] == 0).astype(int)

print(f"RFV criado. {len(rfv)} clientes com compras")

RFV criado. 10000 clientes com compras


## **Features Comportamentais:**

In [3]:
# Compras por categoria
categoria_compras = transacoes.groupby(['id_cliente', 'categoria']).size().unstack(fill_value=0)
categoria_compras.columns = [f'compras_{col.lower()}' for col in categoria_compras.columns]
categoria_compras = categoria_compras.reset_index()

# Método de pagamento preferido
pagamento_pref = transacoes.groupby(['id_cliente', 'metodo_pagamento']).size().unstack(fill_value=0)
pagamento_pref['metodo_preferido'] = pagamento_pref.idxmax(axis=1)
pagamento_pref = pagamento_pref[['metodo_preferido']].reset_index()

# Primeira e última compra
tempo_compras = transacoes.groupby('id_cliente')['data_compra'].agg(['min', 'max']).reset_index()
tempo_compras.columns = ['id_cliente', 'primeira_compra', 'ultima_compra']
tempo_compras['dias_entre_compras'] = (tempo_compras['ultima_compra'] - tempo_compras['primeira_compra']).dt.days
tempo_compras['dias_entre_compras'] = tempo_compras['dias_entre_compras'].fillna(0)

# Fazer merge de todas as features
df = df.merge(categoria_compras, on='id_cliente', how='left')
df = df.merge(pagamento_pref, on='id_cliente', how='left')
df = df.merge(tempo_compras[['id_cliente', 'dias_entre_compras']], on='id_cliente', how='left')

# Preencher valores para clientes sem compras
categorias = ['compras_eletrônicos', 'compras_moda', 'compras_casa', 'compras_esportes', 'compras_beleza']
for cat in categorias:
    if cat in df.columns:
        df[cat] = df[cat].fillna(0)
    else:
        df[cat] = 0

df['metodo_preferido'] = df['metodo_preferido'].fillna('Sem compras')
df['dias_entre_compras'] = df['dias_entre_compras'].fillna(0)

## **Features de Proporção:**

In [4]:
# Proporção por categoria
for cat in categorias:
    prop_col = f'prop_{cat}'
    df[prop_col] = df[cat] / df['frequencia_compras'].replace(0, np.nan)
    df[prop_col] = df[prop_col].fillna(0)

# Ticket médio relativo
media_ticket = df['ticket_medio'].mean()
df['ticket_medio_relativo'] = df['ticket_medio'] / media_ticket


## **Features Temporais:**

In [5]:
# Mês de cadastro 
if 'data_cadastro' in df.columns:
    df['data_cadastro'] = pd.to_datetime(df['data_cadastro'])
    df['mes_cadastro'] = df['data_cadastro'].dt.month
    df['ano_cadastro'] = df['data_cadastro'].dt.year

## **Preparação para Modelagem:**

In [6]:
# Features para o modelo
features_numericas = [
    'idade', 'recencia_dias', 'frequencia_compras', 'valor_total',
    'ticket_medio', 'std_valor', 'dias_entre_compras', 'ticket_medio_relativo',
    'cliente_sem_compra'
] + categorias + [f'prop_{cat}' for cat in categorias]

# Features categóricas
features_categoricas = ['genero', 'cidade', 'canal_aquisicao', 'metodo_preferido']

# One-hot encoding para categóricas
df_encoded = pd.get_dummies(df[features_categoricas], prefix=features_categoricas, drop_first=True)

# Combinar features
X = pd.concat([df[features_numericas], df_encoded], axis=1)
y = df['churn']

# Escalar features numéricas
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns
)

print(f"\nFeatures finais: {X_scaled.shape[1]}")

# Verificar correlações
print("\nTOP 10 CORRELAÇÕES COM CHURN:")
correlacoes = pd.DataFrame({
    'feature': X_scaled.columns,
    'correlacao': [abs(X_scaled[col].corr(y)) for col in X_scaled.columns]
}).sort_values('correlacao', ascending=False)

print(correlacoes.head(10).to_string(index=False))

# Salvar
X_scaled.to_csv('../data/processed/X_features.csv', index=False)
y.to_csv('../data/processed/y_target.csv', index=False)
df.to_csv('../data/processed/clientes_com_features.csv', index=False)

print("\nDados salvos em data/processed/")


Features finais: 29

TOP 10 CORRELAÇÕES COM CHURN:
                  feature  correlacao
       frequencia_compras    0.257078
              valor_total    0.219120
             compras_moda    0.167273
canal_aquisicao_Indicação    0.159496
                cidade_SP    0.149506
       dias_entre_compras    0.135450
             compras_casa    0.123477
         compras_esportes    0.121623
      compras_eletrônicos    0.120412
           compras_beleza    0.118304


C:\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]



Dados salvos em data/processed/
